# EPFL Machine Learning (CS-433) — Project 1
## End-to-End Sample Solution & Educational Walkthrough: Myocardial Infarction Risk Prediction
**Dataset:** CDC Behavioral Risk Factor Surveillance System (BRFSS 2015)  
**Goal:** Predict coronary heart disease / myocardial infarction risk (`_MICHD` in {-1, +1})  
**Constraints:** Only NumPy, Python standard library, and Matplotlib/Seaborn. No external ML libraries (no scikit-learn, no pandas, etc.).

---

### Notebook Architecture:
1. **Exploratory Data Analysis (EDA):** Target imbalance, missingness patterns, and feature correlations.
2. **Preprocessing Pipeline:** High-NaN filtering, missingness indicators, median imputation, standardization, intercept.
3. **Core ML Implementations:** Verification of the 6 required algorithms from class.
4. **Stratified Cross-Validation & Threshold Tuning:** Addressing severe 8.83% class imbalance.
5. **Model Benchmarking & Ablation Study:** Least Squares vs. Ridge Regression vs. Logistic Regression.
6. **Generating Final Submission:** Exporting `submission.csv` via the official course helper.


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Local project imports
import implementations as imp
from helpers import create_csv_submission
from src.preprocessing import PreprocessingPipeline, load_raw_dataset
from src.models import LeastSquaresModel, RidgeRegressionModel, LogisticRegressionModel, RegularizedLogisticRegressionModel
from src.metrics import evaluate_predictions, find_optimal_threshold, compute_f1_score
from src.cross_validation import build_k_fold_indices, cross_validate_pipeline

%matplotlib inline
np.random.seed(42)
print("All modules loaded successfully.")


## 1. Exploratory Data Analysis (EDA)

Let us load a 50,000-sample subset to inspect feature statistics, class distributions, and missing value patterns without excessive memory overhead.


In [ ]:
data_dir = os.path.join("data", "dataset")
X_sample, y_sample, _, train_ids, _ = load_raw_dataset(data_dir, max_samples=50000)

n_pos = np.sum(y_sample == 1)
n_neg = np.sum(y_sample == -1)
print(f"Loaded sample: {X_sample.shape[0]} rows, {X_sample.shape[1]} raw features")
print(f"Class Distribution: {n_pos} positive ({n_pos/len(y_sample):.2%}) vs {n_neg} negative ({n_neg/len(y_sample):.2%})")


In [ ]:
nan_ratios = np.isnan(X_sample).mean(axis=0)

plt.figure(figsize=(8, 4))
plt.hist(nan_ratios, bins=30, color='#1f77b4', edgecolor='black', alpha=0.7)
plt.axvline(0.5, color='red', linestyle='--', label='50% Missingness Drop Threshold')
plt.xlabel('Fraction of Missing Values (NaN) per Feature')
plt.ylabel('Number of Features')
plt.title('Distribution of Missingness Across 321 BRFSS Features')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

n_dropped = np.sum(nan_ratios > 0.5)
n_kept = np.sum(nan_ratios <= 0.5)
print(f"Features with > 50% missing values: {n_dropped} (will be pruned)")
print(f"Features retained for modeling: {n_kept}")


In [ ]:
# Compute correlations with target for retained features
corrs = []
for j in range(X_sample.shape[1]):
    if nan_ratios[j] <= 0.5:
        col = X_sample[:, j]
        valid = ~np.isnan(col)
        if np.std(col[valid]) > 1e-6:
            c = np.corrcoef(col[valid], y_sample[valid])[0, 1]
            corrs.append((j, c))

corrs.sort(key=lambda x: abs(x[1]), reverse=True)
print("Top 10 most correlated features with _MICHD:")
for idx, c in corrs[:10]:
    print(f"Feature index {idx:3d}: correlation = {c:+.4f}")


## 2. Preprocessing & Feature Engineering Pipeline

Our `PreprocessingPipeline` implements:
1. **Column Filtering:** Drops features with > 50% missing values.
2. **Missingness Indicator Expansion:** Adds a binary column for each feature with missing entries.
3. **Median Imputation:** Replaces remaining missing values with the training feature median.
4. **Variance Filtering:** Drops constant/near-zero variance features.
5. **Standardization:** Zero-mean, unit-variance scaling.
6. **Bias Intercept:** Prepends column of ones.


In [ ]:
pipeline = PreprocessingPipeline(
    max_missing_ratio=0.5,
    add_missing_indicators=True,
    standardize=True,
    add_bias=True
)

tx_sample = pipeline.fit_transform(X_sample)
print(f"Transformed feature matrix: {X_sample.shape[1]} raw -> {tx_sample.shape[1]} processed features")


## 3. Core Algorithm Implementations

Here we verify that our `implementations.py` functions conform strictly to the required signatures:
- `mean_squared_error_gd(y, tx, initial_w, max_iters, gamma)`
- `mean_squared_error_sgd(y, tx, initial_w, max_iters, gamma)`
- `least_squares(y, tx)`
- `ridge_regression(y, tx, lambda_)`
- `logistic_regression(y, tx, initial_w, max_iters, gamma)`
- `reg_logistic_regression(y, tx, lambda_, initial_w, max_iters, gamma)`


In [ ]:
# Quick sanity check on sample data
w_ls, loss_ls = imp.least_squares(y_sample[:1000], tx_sample[:1000])
w_rr, loss_rr = imp.ridge_regression(y_sample[:1000], tx_sample[:1000], lambda_=0.01)

print(f"Least squares sample loss: {loss_ls:.4f}, weight norm: {np.linalg.norm(w_ls):.4f}")
print(f"Ridge regression sample loss: {loss_rr:.4f}, weight norm: {np.linalg.norm(w_rr):.4f}")


## 4. Stratified Cross-Validation & Decision Threshold Tuning

Because positive cases constitute only 8.83% of the dataset, classifying via default continuous cutoff (y_pred >= 0.0) yields an extreme precision/recall imbalance (F1 < 0.02).  
We tune an optimal decision threshold on validation splits to maximize the clinical harmonic metric F1.


In [ ]:
# Split into train/val
n_tr = 40000
tx_tr, y_tr = tx_sample[:n_tr], y_sample[:n_tr]
tx_va, y_va = tx_sample[n_tr:], y_sample[n_tr:]

model = RidgeRegressionModel(lambda_=1e-4)
model.fit(tx_tr, y_tr)
raw_val = model.predict_raw(tx_va)

best_tau, best_f1, taus, f1_scores = find_optimal_threshold(y_va, raw_val, metric="f1", n_thresholds=200)

plt.figure(figsize=(7, 4))
plt.plot(taus, f1_scores, color='#1f77b4', lw=2, label='Validation F1')
plt.axvline(best_tau, color='red', linestyle='--', label=f'Optimal tau = {best_tau:.3f} (F1 = {best_f1:.4f})')
plt.axvline(0.0, color='black', linestyle=':', label='Default tau = 0.0')
plt.xlabel('Decision Threshold tau')
plt.ylabel('F1 Score')
plt.title('Impact of Threshold Selection under Class Imbalance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Compare default vs optimal
m_def = evaluate_predictions(y_va, np.where(raw_val >= 0.0, 1, -1))
m_opt = evaluate_predictions(y_va, np.where(raw_val >= best_tau, 1, -1))

print(f"Default Threshold (0.0):  Acc = {m_def['accuracy']:.4f}, Prec = {m_def['precision']:.4f}, Rec = {m_def['recall']:.4f}, F1 = {m_def['f1']:.4f}")
print(f"Optimal Threshold ({best_tau:.3f}): Acc = {m_opt['accuracy']:.4f}, Prec = {m_opt['precision']:.4f}, Rec = {m_opt['recall']:.4f}, F1 = {m_opt['f1']:.4f}")


## 5. Model Comparison & Ablation Study

Let us compare the cross-validation performance across models:
1. Ridge Regression with optimal threshold
2. Least Squares
3. Ridge Regression with default threshold (tau = 0.0)


In [ ]:
results = {}

# 1. Ridge with optimal threshold
res_ridge = cross_validate_pipeline(lambda: RidgeRegressionModel(lambda_=1e-4), tx_sample, y_sample, k_fold=3, tune_threshold=True)
results['Ridge (Ours)'] = res_ridge['mean_f1']

# 2. Least Squares
res_ls = cross_validate_pipeline(lambda: LeastSquaresModel(), tx_sample, y_sample, k_fold=3, tune_threshold=True)
results['Least Squares'] = res_ls['mean_f1']

# 3. Ridge with default threshold
res_def = cross_validate_pipeline(lambda: RidgeRegressionModel(lambda_=1e-4), tx_sample, y_sample, k_fold=3, tune_threshold=False)
results['Ridge (tau=0)'] = res_def['mean_f1']

print("Cross-Validation Summary:")
for name, score in results.items():
    print(f"{name:20s}: Mean F1 = {score:.4f}")


## 6. Summary & Recommendations for Doing it by Hand

When implementing this project on your own:
1. **Always handle missing values first:** The 321 BRFSS survey features contain substantial missingness. Pruning > 50% missing columns and imputing with medians provides a stable foundation.
2. **Missingness is informative:** Notice how adding binary missing indicators boosts classification F1 score.
3. **Beware of class imbalance:** 8.8% positive cases means raw accuracy is deceptive. Always evaluate F1 score and optimize the classification threshold tau.
4. **Regularization is essential:** Survey indicators create near-colinear features. Unregularized least squares will be ill-conditioned, whereas Ridge regression (X^T X + 2N lambda I)^(-1) is provably invertible and fast.
5. **Execution:** Run `python run.py` to reproduce the final submission on the entire 328k dataset.
